In [ ]:
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

ollamaApiKey = os.getenv("OLLAMA_API_KEY")
ollamaBaseUrl = os.getenv("OLLAMA_BASE_URL")
print(ollamaApiKey, ollamaBaseUrl)


In [ ]:
DEEPSEEK_MODEL = "deepseek-r1:8b"
QWEN_MODEL = "qwen3:14b"
GEMMA_MODEL = "gemma3:12b"
GPT_MODEL = "gpt-oss:20b"

In [ ]:
def generateBody(question):
    return [{"role": "user", "content": question}]


In [ ]:
question = "Give me a question that multiple AI models can answer. I want to use the answers from multiple models and score them based on the response they generate. Give me the question only, no answers."
body = generateBody(question=question)
print(body)

In [ ]:
from openai import OpenAI

In [ ]:
openAiClient = OpenAI(base_url=ollamaBaseUrl, api_key=ollamaApiKey)

In [ ]:
def generateAnswer(model, body):
    response = openAiClient.chat.completions.create(
        model=model,
        messages=body
    )
    return response.choices[0].message.content

In [ ]:
questionToBeAskedToLLM = generateAnswer(model=QWEN_MODEL, body=body)


In [ ]:
print(questionToBeAskedToLLM)

In [ ]:
def parseResponseByRemovingTag(tagName, blob):
    from bs4 import BeautifulSoup
    parsedQuestion = BeautifulSoup(blob, "html.parser")
    thinkTag = parsedQuestion.find(tagName)
    if thinkTag:
        thinkTag.decompose()
    return parsedQuestion.prettify()

In [ ]:
parsedQuestionToBeAskedToLLM = parseResponseByRemovingTag("think", questionToBeAskedToLLM) + "\nAnswer should not be more than 1500 characters."
print(parsedQuestionToBeAskedToLLM)

In [ ]:
bodyForNextLLMs = generateBody(parsedQuestionToBeAskedToLLM)
print(bodyForNextLLMs)

In [ ]:
from IPython.display import Markdown, display

def parseMarkdown(blob):
    return Markdown(blob)

In [ ]:
gemmaAnswer = generateAnswer(model=GEMMA_MODEL, body=bodyForNextLLMs)

In [ ]:
print(display(parseMarkdown(gemmaAnswer)))

In [ ]:
deepseekAnswer = generateAnswer(model=DEEPSEEK_MODEL, body=bodyForNextLLMs)

In [ ]:
deepseekAnswerCleaned = parseResponseByRemovingTag("think", deepseekAnswer)
print(display(parseMarkdown(deepseekAnswerCleaned)))

In [ ]:
gptAnswer = generateAnswer(GPT_MODEL, bodyForNextLLMs)

In [ ]:
print(display(parseMarkdown(gptAnswer)))

In [ ]:
nextQuestionForJudge = f"""
This was the inital question I asked: {question}.

You generated this question for LLMs to answer: {parsedQuestionToBeAskedToLLM}
These are the answers to the question you generated.

Answer from GEMMA:

{gemmaAnswer}

Answer from DEEPSEEK:

{deepseekAnswerCleaned}

Answer from GPT:

{gptAnswer}


Now give scores out of 10 based on the answers and rank them. Give in JSON format like {{LLM NAME:  score}}. 
"""

In [ ]:
print(nextQuestionForJudge)

In [ ]:
bodyForJudgement = generateBody(nextQuestionForJudge)

In [ ]:
judgement = generateAnswer(QWEN_MODEL, bodyForJudgement)


In [ ]:
print(display(parseMarkdown(parseResponseByRemovingTag("think", judgement))))